## Minimal reproduction of DIANNA issue #877

This notebook reproduces the silent download failure seen in DIANNA's
`tutorials/conversion_onnx/tensorflow2onnx.ipynb` when running on CI.
Extra diagnostic cells are added to identify exactly where and why it fails.

In [ ]:
import os
import sys
import urllib.request
import urllib.error
import tensorflow as tf

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")

### Diagnostic: test network connectivity

Check whether the CI runner can reach the download URL before we attempt the download.

In [ ]:
DOWNLOAD_URL = (
    'https://storage.googleapis.com/download.tensorflow.org'
    '/models/mobilenet_v1_1.0_224_frozen.tgz'
)

print(f"Testing connectivity to: {DOWNLOAD_URL}")
try:
    req = urllib.request.Request(DOWNLOAD_URL, method='HEAD')
    with urllib.request.urlopen(req, timeout=15) as response:
        print(f"HTTP status  : {response.status}")
        print(f"Content-Type : {response.headers.get('Content-Type', 'n/a')}")
        print(f"Content-Length: {response.headers.get('Content-Length', 'unknown')} bytes")
except urllib.error.URLError as e:
    print(f"URL not reachable: {e}")
except Exception as e:
    print(f"Unexpected error: {type(e).__name__}: {e}")

### Download the MobileNet model

This is the exact code from DIANNA's `tensorflow2onnx.ipynb` notebook.

In [ ]:
print("Calling tf.keras.utils.get_file ...")
fname = tf.keras.utils.get_file(
    'mobilenet.tgz',
    DOWNLOAD_URL,
    extract=True)
print(f"get_file returned: {fname}")

### Diagnostic: verify what was downloaded and extracted

In [ ]:
print(f"Returned path exists : {os.path.exists(fname)}")
if os.path.exists(fname):
    print(f"File size            : {os.path.getsize(fname):,} bytes")

download_dir = os.path.dirname(fname)
print(f"\nDownload directory   : {download_dir}")
if os.path.isdir(download_dir):
    entries = sorted(os.listdir(download_dir))
    print(f"Directory contents ({len(entries)} entries):")
    for entry in entries:
        entry_path = os.path.join(download_dir, entry)
        if os.path.isfile(entry_path):
            print(f"  [file] {entry}  ({os.path.getsize(entry_path):,} bytes)")
        else:
            print(f"  [dir]  {entry}")
else:
    print("Download directory does not exist!")

# Check the extracted MobileNet subdirectory
mobilenet_dir = os.path.join(download_dir, 'mobilenet_v1_1.0_224')
print(f"\nExtracted subdir     : {mobilenet_dir}")
print(f"Extracted subdir exists: {os.path.isdir(mobilenet_dir)}")
if os.path.isdir(mobilenet_dir):
    for entry in sorted(os.listdir(mobilenet_dir)):
        entry_path = os.path.join(mobilenet_dir, entry)
        size = os.path.getsize(entry_path) if os.path.isfile(entry_path) else ''
        print(f"  {entry}" + (f"  ({size:,} bytes)" if size != '' else ''))

### Load the frozen graph

This cell is the one that fails in DIANNA's CI (FileNotFoundError or similar)
when the download/extraction above did not succeed.

In [ ]:
graph_file = os.path.join(os.path.dirname(fname), 'mobilenet_v1_1.0_224/frozen_graph.pb')
print(f"Opening: {graph_file}")

graph_def = tf.compat.v1.GraphDef()
with open(graph_file, 'rb') as f:
    graph_def.ParseFromString(f.read())

print(f"Graph loaded successfully ({len(graph_def.node)} nodes)")